In [27]:
# Cell 1: imports & basic config

import os
import re
from pathlib import Path
import json

import docx  # for reading .docx
import pandas as pd
from openai import OpenAI  # new-style OpenAI client

# set up OpenAI client (expects OPENAI_API_KEY in env)
client = OpenAI()

TRANSCRIPTS_DIR = Path("cleantranscripts")

# choose the model you want to use
MODEL_NAME = "gpt-4.1-mini"  # or another responses-capable model


In [28]:
# Cell 2: loading .docx transcripts

def read_docx_text(path: Path) -> str:
    doc = docx.Document(path)
    paragraphs = [p.text for p in doc.paragraphs]
    text = "\n".join(paragraphs)
    # normalize newlines
    text = re.sub(r'\r', '\n', text)
    text = re.sub(r'\n{2,}', '\n\n', text)
    return text

def load_all_transcripts(transcripts_dir: Path = TRANSCRIPTS_DIR):
    """Yield (file_id, full_text) for each .docx in transcripts_dir."""
    for path in sorted(transcripts_dir.glob("*.docx")):
        file_id = path.stem  # e.g. '19857_clean'
        yield file_id, read_docx_text(path)

# quick smoke-test
for file_id, txt in load_all_transcripts():
    print("Loaded:", file_id, "len:", len(txt))
    break


Loaded: 16234_clean len: 21994


In [ ]:
# Cell 3: No chunking - process full transcript

# Removed chunking logic - we'll send the full transcript text to ChatGPT


In [30]:
EXTRACTION_SYSTEM_PROMPT = """
You are helping a researcher extract structured data from interview transcripts
about kidney allocation decisions.

The transcript contains exactly 3 pairwise comparisons between Patient A and Patient B,
their attributes, and the interviewee's eventual decision about who should receive
the kidney in each scenario.

Your task: read the ENTIRE transcript and output a JSON object with this schema:

{
  "comparisons": [
    {
      "comparison_index": 1,
      "scenario_has_valid_pair": true or false,
      "chosen_patient": "A" | "B" | null,
      "choice_confidence": "high" | "medium" | "low",
      "patient_A": {
        "dependents": int or null,
        "years_life_gained": int or null,
        "obesity_level": "underweight" | "normal" | "overweight" | "obese" | null,
        "work_hours_per_week": int or null,
        "waitlist_years": int or null,
        "serious_criminal_history": true | false | null
      },
      "patient_B": {
        "dependents": int or null,
        "years_life_gained": int or null,
        "obesity_level": "underweight" | "normal" | "overweight" | "obese" | null,
        "work_hours_per_week": int or null,
        "waitlist_years": int or null,
        "serious_criminal_history": true | false | null
      },
      "reasoning_summary": string
    },
    ... (exactly 3 comparisons total)
  ]
}

Important instructions:
- Search through the ENTIRE transcript text to find all 3 pairwise comparisons.
- Extract exactly 3 comparisons, numbered 1, 2, and 3 in order of appearance.
- If a field is not mentioned or you're uncertain, use null for numbers/booleans
  and null for obesity_level.
- "serious_criminal_history" should be:
    - false if the text clearly states the person has NOT committed serious crimes.
    - true if they clearly HAVE committed serious crimes.
    - null if unclear.
- "chosen_patient" should be:
    - "A" if the interviewee clearly leans toward or chooses Patient A.
    - "B" if they clearly lean toward or choose Patient B.
    - null if they explicitly refuse to choose or the comparison does not include a decision.
- "scenario_has_valid_pair" is false if a comparison does not contain a clear A/B comparison.
- "comparison_index" should be 1, 2, or 3 based on the order they appear in the transcript.

Output MUST be valid JSON only, with double quotes and no trailing commas.
Do not include any explanation outside the JSON.
"""

def extract_all_comparisons_with_llm(full_text: str) -> dict:
    """
    Send the entire transcript text to ChatGPT and get back all 3 pairwise comparisons.
    """
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": EXTRACTION_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": full_text,
            },
        ],
        response_format={"type": "json_object"},  # ask for JSON
    )

    # content is a JSON string
    content = response.choices[0].message.content

    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        data = {
            "comparisons": []
        }
    return data


In [31]:
# Cell 5: main loop: transcripts -> full text -> LLM extraction (all 3 comparisons)

all_rows = []

for file_id, txt in load_all_transcripts():
    print(f"{file_id}: Processing full transcript (length: {len(txt)} chars)")

    # Extract all 3 comparisons from the full transcript
    data = extract_all_comparisons_with_llm(txt)
    
    comparisons = data.get("comparisons", [])
    print(f"{file_id}: Found {len(comparisons)} comparisons")

    for comp in comparisons:
        comparison_index = comp.get("comparison_index")
        
        # skip if LLM thinks this isn't a valid A/B scenario
        if not comp.get("scenario_has_valid_pair", False):
            continue

        # flatten for DataFrame
        row = {
            "transcript_id": file_id,
            "comparison_index": comparison_index,
            "chosen_patient": comp.get("chosen_patient"),
            "choice_confidence": comp.get("choice_confidence"),
            "reasoning_summary": comp.get("reasoning_summary"),
        }

        # patient A and B fields
        pa = comp.get("patient_A", {}) or {}
        pb = comp.get("patient_B", {}) or {}
        for key in ["dependents", "years_life_gained", "obesity_level",
                    "work_hours_per_week", "waitlist_years", "serious_criminal_history"]:
            row[f"A_{key}"] = pa.get(key)
            row[f"B_{key}"] = pb.get(key)

        all_rows.append(row)

len(all_rows)


16234_clean: Processing full transcript (length: 21994 chars)
16234_clean: Found 3 comparisons
19857_clean: Processing full transcript (length: 27585 chars)
19857_clean: Found 3 comparisons
23405_clean: Processing full transcript (length: 38415 chars)
23405_clean: Found 3 comparisons
25171_clean: Processing full transcript (length: 33601 chars)
25171_clean: Found 3 comparisons
28789_clean: Processing full transcript (length: 25864 chars)
28789_clean: Found 3 comparisons
31772_clean: Processing full transcript (length: 39223 chars)
31772_clean: Found 3 comparisons
42828_clean: Processing full transcript (length: 34719 chars)
42828_clean: Found 3 comparisons
45865_clean: Processing full transcript (length: 37738 chars)
45865_clean: Found 3 comparisons
51283_clean: Processing full transcript (length: 30345 chars)
51283_clean: Found 3 comparisons
53826_clean: Processing full transcript (length: 39240 chars)
53826_clean: Found 3 comparisons
54338_clean: Processing full transcript (length: 2

60

In [34]:
# Save all_rows to JSON file
output_file = "interview_comparisons.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_rows, f, indent=2, ensure_ascii=False)

print(f"Saved {len(all_rows)} rows to {output_file}")

# Also display the data
all_rows

Saved 60 rows to interview_comparisons.json


[{'transcript_id': '16234_clean',
  'comparison_index': 1,
  'chosen_patient': 'B',
  'choice_confidence': 'high',
  'reasoning_summary': 'Chosen patient B because they have been on the waiting list much longer, and although they gain fewer years of life from the transplant, they still have 10 years left, making it the best choice given all else equal.',
  'A_dependents': 0,
  'B_dependents': 0,
  'A_years_life_gained': 25,
  'B_years_life_gained': 10,
  'A_obesity_level': 'normal',
  'B_obesity_level': 'normal',
  'A_work_hours_per_week': 20,
  'B_work_hours_per_week': 20,
  'A_waitlist_years': 1,
  'B_waitlist_years': 7,
  'A_serious_criminal_history': False,
  'B_serious_criminal_history': False},
 {'transcript_id': '16234_clean',
  'comparison_index': 2,
  'chosen_patient': 'B',
  'choice_confidence': 'medium',
  'reasoning_summary': 'Chose patient B mainly because of longer waitlist time, despite being overweight. The interviewee was uncertain about the impact of obesity but consi